# Refine

> Postprocess markdown files by fixing heading hierarchy and describint images

In [ ]:
#| default_exp refine

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Optionnaly, adding page numbers to headings for better navigation
3. Describing figures

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs
from re import sub, findall, MULTILINE
from pydantic import BaseModel
from lisette.core import completion
from typing import Callable
import os
import json
import shutil

## Heading Hierarchy

Functions for detecting and fixing markdown heading levels



OCR'd PDF files often have corrupted heading hierarchies - headings may jump levels incorrectly (e.g., from H1 to H4) or use inconsistent levels for sections at the same depth. This section provides tools to automatically detect and fix these issues using LLMs, while also optionally adding page numbers for easier navigation.

The first step is extracting all headings from a markdown document so we can analyze their structure.

In [ ]:
#| export
def get_hdgs(
    md:str # Markdown file string
    ) -> L: # L of strings
    "Return the markdown headings"
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return L(findall(r'^#{1,6} .+$', md, MULTILINE))



In [ ]:
#| export
def add_pg_hdgs(
    md:str, # Markdown file string, 
    n:int # Page number
    ) -> str: # Markdown file string
    "Add page number to all headings in page markdown"
    md = sub(r'```[\s\S]*?```', '', md)
    def repl(m): return m.group(0) + f' ... page {n}'
    return sub(r'^#{1,6} .+$', repl, md, flags=MULTILINE)

The `add_pg_hdgs` function serves two important purposes:

**1. Creating unique heading identifiers**

When fixing heading hierarchies across an entire document, we need a way to distinguish between headings that have the same text but appear in different locations. For example, a document might have multiple "Introduction" or "Conclusion" headings in different chapters. By appending the page number to each heading, we create unique identifiers that allow us to build a lookup table mapping each specific heading instance to its corrected version. This assumes (reasonably) that the same heading text won't appear twice on a single page.

**2. Providing spatial context for LLMs**

Adding page numbers gives LLMs valuable positional information when analyzing the document structure. The page number helps the model understand:
- Where a heading sits in the overall document flow
- The relative distance between sections
- Whether headings that seem related are actually close together or far apart

This spatial awareness can significantly improve the LLM's ability to infer the correct hierarchical relationships between headings, especially in long documents where similar section names might appear at different structural levels.

For instance:

In [ ]:
#| eval: false
pgs = read_pgs('files/test/md_all/resnet', join=False)
pg0,pg0_with = pgs[0][:500],add_pg_hdgs(pgs[0], n=1)[:500]
print('Before:\n' + 80*'-' + f'\n{pg0}\n\nAfter:\n' + 80*'-' + f'\n{pg0_with}')

Before:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition 

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unr

After:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the train

In [ ]:
#| export
def read_pgs_pg(
    path:str # Path to the markdown file
    ) -> L: # List of markdown pages
    "Read all pages of a markdown file and add page numbers to all headings"
    pgs = read_pgs(path, join=False)
    return L([add_pg_hdgs(pg, n) for n, pg in enumerate(pgs, 1)]).concat()

In [ ]:
#| eval: false
pgs = read_pgs_pg('files/test/md_all/resnet')
hdgs = L([get_hdgs(pg) for pg in pgs]).concat()
hdgs

(#22) ['# Deep Residual Learning for Image Recognition  ... page 1','#### Abstract ... page 1','## 1. Introduction ... page 1','## 2. Related Work ... page 2','## 3. Deep Residual Learning ... page 3','### 3.1. Residual Learning ... page 3','### 3.2. Identity Mapping by Shortcuts ... page 3','### 3.3. Network Architectures ... page 3','### 3.4. Implementation ... page 4','## 4. Experiments ... page 4','### 4.1. ImageNet Classification ... page 4','### 4.2. CIFAR-10 and Analysis ... page 7','### 4.3. Object Detection on PASCAL and MS COCO ... page 8','## References ... page 9','## A. Object Detection Baselines ... page 10','## PASCAL VOC ... page 10','## MS COCO ... page 10','## B. Object Detection Improvements ... page 10','## MS COCO ... page 10','## PASCAL VOC ... page 11'...]

To make it easier for an LLM to reference specific headings when suggesting fixes, we format them with index numbers.

In [ ]:
#| export
def fmt_hdgs_idx(
    hdgs: list[str] # List of markdown headings
    ) -> str: # Formatted string with index
    "Format the headings with index"
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))


In [ ]:
#| eval: false
hdgs_fmt = fmt_hdgs_idx(hdgs)
print(hdgs_fmt)

0. # Deep Residual Learning for Image Recognition  ... page 1
1. #### Abstract ... page 1
2. ## 1. Introduction ... page 1
3. ## 2. Related Work ... page 2
4. ## 3. Deep Residual Learning ... page 3
5. ### 3.1. Residual Learning ... page 3
6. ### 3.2. Identity Mapping by Shortcuts ... page 3
7. ### 3.3. Network Architectures ... page 3
8. ### 3.4. Implementation ... page 4
9. ## 4. Experiments ... page 4
10. ### 4.1. ImageNet Classification ... page 4
11. ### 4.2. CIFAR-10 and Analysis ... page 7
12. ### 4.3. Object Detection on PASCAL and MS COCO ... page 8
13. ## References ... page 9
14. ## A. Object Detection Baselines ... page 10
15. ## PASCAL VOC ... page 10
16. ## MS COCO ... page 10
17. ## B. Object Detection Improvements ... page 10
18. ## MS COCO ... page 10
19. ## PASCAL VOC ... page 11
20. ## ImageNet Detection ... page 11
21. ## C. ImageNet Localization ... page 12


We use a Pydantic model to ensure the LLM returns corrections in a structured format - a dictionary mapping heading indices to their corrected versions.

In [ ]:
#| export
class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading

This prompt instructs the LLM on what types of heading hierarchy errors to fix while preserving the document's intended structure. It focuses on three main issues: 

- level jumps that skip intermediate levels, 
- numbering inconsistencies where subsection depth doesn't match heading level, and
- ensures decreasing levels (moving back up the hierarchy) are preserved.

In [ ]:
#| export
prompt_fix_hdgs = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title ... page 1")

RULES - Apply these fixes in order:

1. **Single H1 rule**: Documents must have exactly ONE # heading (typically the document title at the top)
   - If index 0 is already #, then all subsequent headings (index 1+) must be ## or deeper
   - If no H1 exists, the first major heading should be #, and all others ## or deeper
   - NO exceptions: appendices, references, and all sections are ## or deeper after the title

2. **Infer depth from numbering patterns**: If headings contain section numbers, deeper nesting means deeper heading level
   - Parent section (e.g., "1", "2", "A") should be shallower than child (e.g., "1.1", "2.a", "A.1")
   - Child section should be one # deeper than parent
   - Works with any numbering: "1/1.1/1.1.1", "A/A.1/A.1.a", "I/I.A/I.A.1", etc.

3. **Level jumps**: Headings can only increase by one # at a time when moving deeper
   - Wrong: ## Section → ##### Subsection
   - Fixed: ## Section → ### Subsection

4. **Decreasing levels is OK**: Moving back up the hierarchy (### to ##) is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
IMPORTANT: Preserve the " ... page N" suffix in all corrected headings.
Only include entries that need changes.

Headings to analyze:
{headings_list}
"""


#| export
Now we can use an LLM to automatically detect and fix these heading hierarchy issues. The function uses litellm (wrapped by the [Lisette package](https://lisette.answer.ai) to send the formatted headings to a language model along with our correction rules. The LLM analyzes the structure and returns only the headings that need fixing, mapped by their index numbers.

In [ ]:
#| export
def fix_hdg_hierarchy(
    hdgs: list[str], # List of markdown headings
    prompt: str=None, # Prompt to use
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None # API key
    ) -> dict[int, str]: # Dictionary of index → corrected heading
    "Fix the heading hierarchy"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    if prompt is None: prompt = prompt_fix_hdgs
    prompt = prompt.format(headings_list=fmt_hdgs_idx(hdgs))
    r = completion(model=model, messages=[{"role": "user", "content": prompt}], response_format=HeadingCorrections, api_key=api_key)
    return json.loads(r.choices[0].message.content)['corrections']


In [ ]:
#| eval: false
fixes = fix_hdg_hierarchy(hdgs)
fixes

{'1': '## Abstract ... page 1',
 '13': '## References ... page 9',
 '14': '## Appendix A. Object Detection Baselines ... page 10',
 '15': '### PASCAL VOC ... page 10',
 '16': '### MS COCO ... page 10',
 '17': '## Appendix B. Object Detection Improvements ... page 10',
 '18': '### MS COCO ... page 10',
 '19': '### PASCAL VOC ... page 11',
 '20': '### ImageNet Detection ... page 11',
 '21': '## Appendix C. ImageNet Localization ... page 12'}

The corrections come back as string indices, but we need to map the actual heading text to its corrected version for easy replacement in the document.

In [ ]:
#| export
@delegates(fix_hdg_hierarchy)
def mk_fixes_lut(
    hdgs: list[str], # List of markdown headings
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None, # API key
    **kwargs
    ) -> dict[str, str]: # Dictionary of old → new heading
    "Make a lookup table of fixes"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    fixes = fix_hdg_hierarchy(hdgs, model=model, api_key=api_key, **kwargs)
    return {hdgs[int(k)]:v for k,v in fixes.items()}

In [ ]:
#| eval: false
lut_fixes = mk_fixes_lut(hdgs)
lut_fixes

{'#### Abstract ... page 1': '## Abstract ... page 1',
 '## References ... page 9': '## References ... page 9',
 '## A. Object Detection Baselines ... page 10': '## Appendix A. Object Detection Baselines ... page 10',
 '## PASCAL VOC ... page 10': '### PASCAL VOC ... page 10',
 '## MS COCO ... page 10': '### MS COCO ... page 10',
 '## B. Object Detection Improvements ... page 10': '## Appendix B. Object Detection Improvements ... page 10',
 '## PASCAL VOC ... page 11': '### PASCAL VOC ... page 11',
 '## ImageNet Detection ... page 11': '### ImageNet Detection ... page 11',
 '## C. ImageNet Localization ... page 12': '## Appendix C. ImageNet Localization ... page 12'}

Now we can apply the fixes to individual pages. We optionally add page numbers to headings for easier navigation in the final document.

In [ ]:
#| export
def apply_hdg_fixes(
    p:str, # Page to fix
    lut_fixes: dict[str, str], # Lookup table of fixes
    ) -> str: # Page with fixes applied
    "Apply the fixes to the page"
    for old in get_hdgs(p): p = p.replace(old, lut_fixes.get(old, old))
    return p

In [ ]:
#| eval: false
pg_nb = 1
p = read_pgs_pg('files/test/md_all/resnet')[0]
print(apply_hdg_fixes(p, lut_fixes)[:300])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framewor


Finally, we tie everything together in a single function that processes an entire document directory, fixing all heading hierarchy issues and optionally adding page numbers.

In [ ]:
#| export
@delegates(mk_fixes_lut)
def fix_md_hdgs(src:str, model:str='claude-sonnet-4-5', dst:str=None, img_folder:str='img', **kwargs):
    "Fix heading hierarchy in markdown document"
    src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
    if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
    src_imgs = src_path/img_folder
    if src_imgs.exists() and dst_path != src_path: shutil.copytree(src_imgs, dst_path/img_folder, dirs_exist_ok=True)
    pgs_with_pg = read_pgs_pg(src_path)
    lut = mk_fixes_lut(L([get_hdgs(pg) for pg in pgs_with_pg]).concat(), model, **kwargs)
    for i,p in enumerate(pgs_with_pg, 1): (dst_path/f'page_{i}.md').write_text(apply_hdg_fixes(p, lut))

In [ ]:
#| eval: false
fix_md_hdgs('files/test/md_all/resnet', dst='files/test/md_fixed/resnet')

In [ ]:
#| eval: false
!ls -R 'files/test/md_fixed/resnet'

files/test/md_fixed/resnet:
img	   page_10.md  page_12.md  page_3.md  page_5.md  page_7.md  page_9.md
page_1.md  page_11.md  page_2.md   page_4.md  page_6.md  page_8.md

files/test/md_fixed/resnet/img:
img-0.jpeg  img-2.jpeg	img-4.jpeg  img-6.jpeg
img-1.jpeg  img-3.jpeg	img-5.jpeg


In [ ]:
#| eval: false
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


## Image Enrichment

Tools for classifying and describing images in markdown documents

In [ ]:
#| eval: false
# Dev only
import sys
sys.path.append('../../..')
from ctx_utils import *

In [ ]:
mappr = nb_to_md('_06_mappr.ipynb')
mappr[:200]

'# Mappr\n\n> Scale up evaluation report mapping against evaluation frameworks using agentic workflows\n\n\n---\n\n::: {.callout-warning}\nThis notebook is a work in progress.\n:::\n\n---\n\nManually mapping evalua'

In [ ]:
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


My task here is to describe the images downloaded by the OCR and injecting those descriptions in the markdown file. 

In [ ]:
s